In [1]:
# %% [markdown]
# # Part 2: RFM Segmentation & Targeted Retention Engine
# **Snapshot Cutoff Date:** 2025-09-30

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure reproducibility
np.random.seed(42)

# %% [markdown]
# ### Step 1: Load and Purge Post-Snapshot Order Data (Leakage Guard)

# %%
# In your local runtime, point these to your actual files:
# df_orders = pd.read_csv("orders.csv")
# df_tickets = pd.read_csv("support_tickets.csv")
# df_web = pd.read_csv("web_events_snapshot.csv")

# --- SYNTHESIZE MOCK DATA DIRECTLY MATCHING THE PROVIDED DICTIONARY SCHEMAS ---
customers_universe = [f"CUST{i:05d}" for i in range(1, 2401)]

# Synthesize orders spanning pre- and post-snapshot ranges
order_dates = pd.date_range(start="2024-01-09", end="2025-11-29", periods=10009)
df_orders = pd.DataFrame({
    "order_id": [f"ORD{i:06d}" for i in range(1, 10010)],
    "customer_id": np.random.choice(customers_universe, 10009),
    "order_date": np.random.choice(order_dates, 10009),
    "quantity": np.random.randint(1, 5, 10009),
    "gross_amount": np.random.uniform(149.0, 3500.0, 10009),
    "discount_pct": np.random.uniform(0.0, 0.7, 10009),
    "returned": np.random.choice([0, 1], 10009, p=[0.88, 0.12])
})
# Inject intentional outlier to replicate known data quality issue noted in dictionary
df_orders.loc[42, "gross_amount"] = 24789.38

# Synthesize tickets schema
ticket_dates = pd.date_range(start="2024-01-13", end="2025-09-30", periods=1921)
df_tickets = pd.DataFrame({
    "ticket_id": [f"TKT{i:06d}" for i in range(1, 1922)],
    "customer_id": np.random.choice(customers_universe, 1921),
    "ticket_date": np.random.choice(ticket_dates, 1921),
    "issue_type": np.random.choice(["damaged_item", "late_delivery", "wrong_item", "general_query"], 1921)
})

# Synthesize web snapshot schema
df_web = pd.DataFrame({
    "customer_id": customers_universe,
    "abandoned_carts_30d": np.random.randint(0, 6, 2400),
    "last_visit_days_ago": np.random.randint(0, 45, 2400)
})

# --- ENFORCE CRITICAL DATA LEAKAGE RULE ---
SNAPSHOT_DATE = pd.to_datetime("2025-09-30")
df_orders["order_date"] = pd.to_datetime(df_orders["order_date"])
df_orders_safe = df_orders[df_orders["order_date"] <= SNAPSHOT_DATE].copy()

print(f"Purged {len(df_orders) - len(df_orders_safe)} rows occurring after snapshot date.")

# %% [markdown]
# ### Step 2: Extract RFM and Non-RFM Behavioral Core Aggregates

# %%
# Deduplicate order rows ending in _DUP if present
df_orders_safe = df_orders_safe[~df_orders_safe["order_id"].str.contains("_DUP", na=False)]

# Calculate foundational RFM components
rfm_base = df_orders_safe.groupby("customer_id").agg(
    recency_days=("order_date", lambda x: (SNAPSHOT_DATE - x.max()).days),
    frequency=("order_id", "count"),
    monetary=("gross_amount", "sum"),
    total_returned_orders=("returned", "sum")
).reset_index()

# Extract Non-RFM Signal 1: Order Return Rates
rfm_base["return_rate"] = (rfm_base["total_returned_orders"] / rfm_base["frequency"]).fillna(0)

# Extract Non-RFM Signal 2: Support Ticket Counts
ticket_counts = df_tickets.groupby("customer_id").size().to_frame("ticket_count").reset_index()

# Left merge with customer universe to preserve full 2400 records
base_cohort = pd.DataFrame({"customer_id": customers_universe})
merged_features = base_cohort.merge(rfm_base, on="customer_id", how="left")
merged_features = merged_features.merge(ticket_counts, on="customer_id", how="left")
merged_features = merged_features.merge(df_web, on="customer_id", how="left")

# Fill missing transaction and ticket profiles with zero variables
merged_features["recency_days"] = merged_features["recency_days"].fillna(999) # Max inactivity flag
merged_features["frequency"] = merged_features["frequency"].fillna(0)
merged_features["monetary"] = merged_features["monetary"].fillna(0.0)
merged_features["return_rate"] = merged_features["return_rate"].fillna(0.0)
merged_features["ticket_count"] = merged_features["ticket_count"].fillna(0)

# %% [markdown]
# ### Step 3: Compute Relative Quantile Scoring Maps

# %%
# Score ranges 1-5 where high scores equal best performance metrics
merged_features["R_score"] = pd.qcut(merged_features["recency_days"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
merged_features["F_score"] = pd.qcut(merged_features["frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
merged_features["M_score"] = pd.qcut(merged_features["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

# %% [markdown]
# ### Step 4: Map Behavioral Criteria to 7 Segments

# %%
def allocate_segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]
    tickets = row["ticket_count"]
    ret_rate = row["return_rate"]
    carts = row["abandoned_carts_30d"]

    # Custom Non-RFM Profile 1: High-Value But Unhappy (High Value + Service Friction)
    if m >= 4 and f >= 3 and tickets >= 2:
        return "high_value_unhappy"

    # Custom Non-RFM Profile 2: High-Intent Cart Abandoner (High Web Intent + Low Transaction Conversion)
    if f <= 2 and carts >= 3:
        return "high_intent_abandoner"

    # Core RFM Profiles
    if r >= 4 and f >= 4 and m >= 4:
        return "champions"
    elif r >= 3 and f >= 3 and m >= 3:
        return "loyal_customers"
    elif r <= 2 and f >= 3 and m >= 3:
        return "at_risk_customers"
    elif r >= 4 and f <= 2:
        return "new_customers"
    elif r <= 2 and f <= 2:
        return "dormant_customers"
    else:
        return "discount_sensitive_customers"

merged_features["segment_name"] = merged_features.apply(allocate_segment, axis=1)

# %% [markdown]
# ### Step 5: Save Production Ready Output Table

# %%
export_cols = [
    "customer_id", "recency_days", "frequency", "monetary",
    "ticket_count", "return_rate", "abandoned_carts_30d",
    "R_score", "F_score", "M_score", "segment_name"
]
merged_features[export_cols].to_csv("segments.csv", index=False)
print(merged_features["segment_name"].value_counts())

Purged 880 rows occurring after snapshot date.
segment_name
high_intent_abandoner           488
loyal_customers                 426
champions                       351
at_risk_customers               293
dormant_customers               272
discount_sensitive_customers    270
high_value_unhappy              172
new_customers                   128
Name: count, dtype: int64
